In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/clean_matches.csv")
df.head()

,Season,MatchDate,HomeTeam,AwayTeam,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult
0,2000/01,2000-08-19,Charlton,Man City,0.0,0.0,0.0,0.0,4.0,0.0,H
1,2000/01,2000-08-19,Chelsea,West Ham,0.0,0.0,0.0,0.0,4.0,2.0,H
2,2000/01,2000-08-19,Coventry,Middlesbrough,0.0,0.0,0.0,0.0,1.0,3.0,A
3,2000/01,2000-08-19,Derby,Southampton,0.0,0.0,0.0,0.0,2.0,2.0,D
4,2000/01,2000-08-19,Leeds,Everton,0.0,0.0,0.0,0.0,2.0,0.0,H


# Feature engineering

In [2]:
team_history = df[["MatchDate", "HomeTeam", "AwayTeam", "FTHome", "FTAway"]].copy()

home = team_history[["MatchDate", "HomeTeam", "FTHome", "FTAway"]].copy()
home.columns = ["MatchDate", "Team", "GoalsScored", "GoalsConceded"]

away = team_history[["MatchDate", "AwayTeam", "FTHome", "FTAway"]].copy()
away.columns = ["MatchDate", "Team", "GoalsConceded", "GoalsScored"]

team_history = pd.concat([home, away])
team_history = team_history.sort_values(["Team", "MatchDate"])
team_history.head(10)

,MatchDate,Team,GoalsScored,GoalsConceded
7,2000-08-19,Arsenal,0.0,1.0
10,2000-08-21,Arsenal,2.0,0.0
20,2000-08-26,Arsenal,5.0,3.0
37,2000-09-06,Arsenal,2.0,2.0
39,2000-09-09,Arsenal,1.0,1.0
54,2000-09-16,Arsenal,2.0,1.0
63,2000-09-23,Arsenal,1.0,1.0
76,2000-10-01,Arsenal,1.0,0.0
85,2000-10-14,Arsenal,1.0,0.0
90,2000-10-21,Arsenal,2.0,1.0


In [3]:
team_history["AvgGoalsScored5"] = (
    team_history
    .groupby("Team")["GoalsScored"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_history["AvgGoalsConceded5"] = (
    team_history
    .groupby("Team")["GoalsConceded"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_history.head(20)

,MatchDate,Team,GoalsScored,GoalsConceded,AvgGoalsScored5,AvgGoalsConceded5
7,2000-08-19,Arsenal,0.0,1.0,NaN,NaN
10,2000-08-21,Arsenal,2.0,0.0,NaN,NaN
20,2000-08-26,Arsenal,5.0,3.0,NaN,NaN
37,2000-09-06,Arsenal,2.0,2.0,NaN,NaN
39,2000-09-09,Arsenal,1.0,1.0,NaN,NaN
54,2000-09-16,Arsenal,2.0,1.0,2.0,1.4
63,2000-09-23,Arsenal,1.0,1.0,2.4,1.4
76,2000-10-01,Arsenal,1.0,0.0,2.2,1.6
85,2000-10-14,Arsenal,1.0,0.0,1.4,1.0
90,2000-10-21,Arsenal,2.0,1.0,1.2,0.6


In [4]:
home_stats = team_history[["MatchDate", "Team", "AvgGoalsScored5", "AvgGoalsConceded5"]].copy()
home_stats = home_stats.rename(columns={
    "Team": "HomeTeam",
    "AvgGoalsScored5": "HomeAvgGoalsScored5",
    "AvgGoalsConceded5": "HomeAvgGoalsConceded5",
})

away_stats = team_history[["MatchDate", "Team", "AvgGoalsScored5", "AvgGoalsConceded5"]].copy()
away_stats = away_stats.rename(columns={
    "Team": "AwayTeam",
    "AvgGoalsScored5": "AwayAvgGoalsScored5",
    "AvgGoalsConceded5": "AwayAvgGoalsConceded5",
})

df = df.merge(
    home_stats,
    on=["MatchDate", "HomeTeam"],
    how="left"
)

df = df.merge(
    away_stats,
    on=["MatchDate", "AwayTeam"],
    how="left"
)


In [5]:
df.head(60)

,Season,MatchDate,HomeTeam,AwayTeam,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult,HomeAvgGoalsScored5,HomeAvgGoalsConceded5,AwayAvgGoalsScored5,AwayAvgGoalsConceded5
0,2000/01,2000-08-19,Charlton,Man City,0.0,0.0,0.0,0.0,4.0,0.0,H,NaN,NaN,NaN,NaN
1,2000/01,2000-08-19,Chelsea,West Ham,0.0,0.0,0.0,0.0,4.0,2.0,H,NaN,NaN,NaN,NaN
2,2000/01,2000-08-19,Coventry,Middlesbrough,0.0,0.0,0.0,0.0,1.0,3.0,A,NaN,NaN,NaN,NaN
3,2000/01,2000-08-19,Derby,Southampton,0.0,0.0,0.0,0.0,2.0,2.0,D,NaN,NaN,NaN,NaN
4,2000/01,2000-08-19,Leeds,Everton,0.0,0.0,0.0,0.0,2.0,0.0,H,NaN,NaN,NaN,NaN
5,2000/01,2000-08-19,Leicester,Aston Villa,0.0,0.0,0.0,0.0,0.0,0.0,D,NaN,NaN,NaN,NaN
6,2000/01,2000-08-19,Liverpool,Bradford,0.0,0.0,0.0,0.0,1.0,0.0,H,NaN,NaN,NaN,NaN
7,2000/01,2000-08-19,Sunderland,Arsenal,0.0,0.0,0.0,0.0,1.0,0.0,H,NaN,NaN,NaN,NaN
8,2000/01,2000-08-19,Tottenham,Ipswich,0.0,0.0,0.0,0.0,3.0,1.0,H,NaN,NaN,NaN,NaN
9,2000/01,2000-08-20,Man United,Newcastle,0.0,0.0,0.0,0.0,2.0,0.0,H,NaN,NaN,NaN,NaN


In [6]:
df = df.dropna(subset=[
    "HomeAvgGoalsScored5",
    "HomeAvgGoalsConceded5",
    "AwayAvgGoalsScored5",
    "AwayAvgGoalsConceded5",
])
df.head()

,Season,MatchDate,HomeTeam,AwayTeam,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult,HomeAvgGoalsScored5,HomeAvgGoalsConceded5,AwayAvgGoalsScored5,AwayAvgGoalsConceded5
49,2000/01,2000-09-16,Sunderland,Derby,1.0,4.0,3.0,4.0,2.0,1.0,H,0.8,1.8,2.2,2.4
51,2000/01,2000-09-16,Everton,Man United,4.0,7.0,7.0,11.0,1.0,3.0,A,1.8,1.6,2.8,0.6
52,2000/01,2000-09-16,Southampton,Newcastle,2.0,3.0,7.0,10.0,2.0,0.0,H,1.4,1.8,1.4,0.8
54,2000/01,2000-09-16,Arsenal,Coventry,5.0,8.0,4.0,7.0,2.0,1.0,H,2.0,1.4,1.0,1.4
55,2000/01,2000-09-16,Charlton,Tottenham,2.0,5.0,6.0,10.0,1.0,0.0,H,2.0,2.2,1.6,1.2


## Elo system

In [7]:
import sys
sys.path.append("..")

In [8]:
from src.elo import process_match

ratings = {}

home_elos = []
away_elos = []

for index, match in df.iterrows():
    home_team = match["HomeTeam"]
    away_team = match["AwayTeam"]
    result = match["FTResult"]

    elos = process_match(home_team, away_team, result, ratings)

    old_home_elo, home_elo = elos[0]
    old_away_elo, away_elo = elos[1]

    home_elos.append(old_home_elo)
    away_elos.append(old_away_elo)

print(ratings)

    

{'Sunderland': 1415.004537489091, 'Derby': 1282.0770798042095, 'Everton': 1592.2771824081105, 'Man United': 1570.2790934729474, 'Southampton': 1343.9334759289463, 'Newcastle': 1666.0030684776057, 'Arsenal': 1779.5264760388816, 'Coventry': 1425.7005906584907, 'Charlton': 1455.7643146604703, 'Tottenham': 1527.9557143000886, 'Chelsea': 1689.098703103942, 'Leicester': 1436.0907055000594, 'Man City': 1793.424963054107, 'Middlesbrough': 1409.931346649271, 'West Ham': 1539.3443733814934, 'Liverpool': 1795.5609627976469, 'Aston Villa': 1682.2769880165135, 'Ipswich': 1407.7114333158654, 'Leeds': 1460.1765147419303, 'Bradford': 1400.9778452293858, 'Blackburn': 1445.4786626252958, 'Bolton': 1457.8501595504108, 'Fulham': 1576.2935759123782, 'West Brom': 1431.4928914938935, 'Birmingham': 1480.4113651514874, 'Wolves': 1530.0711502927363, 'Portsmouth': 1426.522427099386, 'Crystal Palace': 1621.9090826466968, 'Norwich': 1352.678536092261, 'Wigan': 1483.6744801047273, 'Reading': 1416.3200286057136, 'Wa

In [9]:
import json

with open("../data/current_elos.json", "w") as f:
    json.dump(ratings, f, indent=4)

In [10]:
df["HomeElo"] = home_elos
df["AwayElo"] = away_elos

df.head()

,Season,MatchDate,HomeTeam,AwayTeam,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult,HomeAvgGoalsScored5,HomeAvgGoalsConceded5,AwayAvgGoalsScored5,AwayAvgGoalsConceded5,HomeElo,AwayElo
49,2000/01,2000-09-16,Sunderland,Derby,1.0,4.0,3.0,4.0,2.0,1.0,H,0.8,1.8,2.2,2.4,1507.1987,1492.8013
51,2000/01,2000-09-16,Everton,Man United,4.0,7.0,7.0,11.0,1.0,3.0,A,1.8,1.6,2.8,0.6,1487.1987,1512.8013
52,2000/01,2000-09-16,Southampton,Newcastle,2.0,3.0,7.0,10.0,2.0,0.0,H,1.4,1.8,1.4,0.8,1507.1987,1492.8013
54,2000/01,2000-09-16,Arsenal,Coventry,5.0,8.0,4.0,7.0,2.0,1.0,H,2.0,1.4,1.0,1.4,1507.1987,1492.8013
55,2000/01,2000-09-16,Charlton,Tottenham,2.0,5.0,6.0,10.0,1.0,0.0,H,2.0,2.2,1.6,1.2,1507.1987,1492.8013


In [ ]:

seasons = [
    "2014/15",
    "2015/16",
    "2016/17",
    "2017/18",
    "2018/19",
    "2019/20",
    "2020/21",
    "2021/22",
    "2022/23",
    "2023/24",
    "2024/25"
]
df = df[df["Season"].isin(seasons)]
df = df.drop(columns=["FTHome", "FTAway", "MatchDate", "HomeTeam", "AwayTeam", "Season"])
df.head()

SyntaxError: invalid syntax (338501221.py, line 15)

In [ ]:
df["EloDifference"] = df["HomeElo"] - df["AwayElo"]
df["FormDifference"] = df["Form5Home"] - df["Form5Away"]

df["AvgGoalsScoredDifference5"] = (
    df["HomeAvgGoalsScored5"] -
    df["AwayAvgGoalsScored5"]
)

df["AvgGoalsConcededDifference5"] = (
    df["HomeAvgGoalsConceded5"] -
    df["AwayAvgGoalsConceded5"]
)

df.head()

In [ ]:
df.to_csv("../data/processed/featured_data.csv", index=False)